In [3]:
import pyorbital
from pyorbital.orbital import Orbital
import datetime as dt
from matplotlib import colormaps as cmap
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pandas as pd
import bisect
import uuid
from enum import Enum

from fame import *
import copy

In [4]:
# Behind the scenes, this pulls from Celestrak if we do not specify tle_file

satellites = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [5]:

ground_stations = [
    Location(
        -79.55,
        8.9833,
        0.028,
        "KSAT Panama",
    ),
    Location(
        -51.73363,
        64.182789,
        0,
        "KSAT Nuuk",
    ),
    Location(
        2.53219,
        -72.01243,
        0,
        "KSAT Troll",
    ),
    Location(
        142.3689,
        43.8,
        0,
        "KSAT Hokkaido",
    ),
    Location(
        103.9915,
        1.3661,
        0,
        "KSAT Singapore",
    ),
    Location(
        -70.85021,
        -52.93279,
        0,
        "KSAT Punta Arenas",
    ),
    Location(
        127.7766,
        26.4055,
        0,
        "KSAT Okinawa",
    ),
    Location(
        57.5565,
        -20.1142,
        0,
        "KSAT Mauritius",
    ),
    Location(
        22.62216,
        37.84604,
        0,
        "KSAT Nemea",
    ),
    Location(
        31.12509,
        70.36779,
        0,
        "KSAT Vardo",
    ),
    Location(
        15.39964,
        78.22875,
        0,
        "KSAT Svalbard",
    ),
]

# ground_station_opportunities = [
#     observation_request(
#         lon_deg=gs.lon_deg,
#         lat_deg=gs.lat_deg,
#         alt_km=gs.alt_km,
#         min_time=dt.datetime.now(dt.timezone.utc),
#         max_time=dt.datetime.now(dt.timezone.utc)+ dt.timedelta(seconds=3600*24*2)
#     )
#     for gs in ground_stations
# ]

In [6]:
stride_s = 60
plot_range_s = 10800

min_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
max_time = min_time + dt.timedelta(hours=24)

In [7]:


cities_of_the_world = pd.read_csv("simplemaps_worldcities_basicv1.901/worldcities.csv")
sampled_world_cities = cities_of_the_world.sample(n=100,weights='population',axis=0, random_state=0)
one_hundred_sampled_cities = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities.iterrows()
]

# Simulation

Let's talk about simulation.

We want an event-based sim. There is a global ordered timeline and the sim jumps from event to event.

Continuous transitions (power, thermal, etc) are discretized in this setting.

We need a few entities here.

Agent: a satellite, a constellation manager, a requestor. 

Event: something on the ground turning on or off.

Communication: something that affects the state of two entities.

Observation: a query of an agent to (event, empty set). If we want to be fancy, a query of an agent to a region, where events are or are 
not associated with regions.

Prototype:
- A Satellite class with a list of Observations
- A ConstellationManager class with a list of Satellites which can query Observations and add new ones.
  - A ConstellationManager class that updates the Satellites' Observations when a Communication event occurs.
- An Observation of a Region.
    - Observe. Returns an image of the region.
    - Search. Returns True if there is an Event in the Region. The event is stored.
    - Monitor. Returns True if the event is still active.
    - Something for moving events?
- Phenomenon: something with a spatial position, start time, end time, potentially internal states.

The simulator holds:
- Agents (Satellites, ConstellationManagers)
- Phenomena.
- A list of Events (Observation, Communication, Phenomenon transitions), encoded as functions.
The output of the functions affects the agents and phenomena.

We need to define:

- An Observation Opportunity (observation_opportunity)
- An Observer, which has an Orbit, a list of Observation Opportunities to execute, a list of ObservedEvents it has observed (with their state), and a list of DataProducts.
- A ConstellationManager, which has a list of ObserverStates (Orbit, ObservationOpportunities) and a list of CommunicationOpportunities with Observers. When there is a CommunicationOpportunity the ConstellationManager can push an updated list of ObservationOpportunities.
- An Event, with a lonlatalt, a State (enum), and times for StateTransitions.
- 

In [8]:
MIN_HORIZON_ANGLE_FOR_PASS_DEG = 15
MIN_HORIZON_ANGLE_FOR_OBS_DEG = 15

class Event():
    def __init__(self, time: dt.datetime, action_callable, name: str="", id: str=None):
        self.time = time
        self.action_callable = action_callable
        self.name = name
        if id is None:
            id = uuid.uuid4()
        self.id = id
    def __str__(self):
        return "Event {} at {}".format(self.name, self.time)
    def __repr__(self):
        return self.__str__()

class ObservationEvent(Event):
    def __init__(self, time: dt.datetime, action_callable, name: str="", id: str=None, satellite: Satellite=None, opportunity: ObservationOpportunity=None):
        super().__init__(time, action_callable, name, id)
        self.satellite = satellite
        self.opportunity = opportunity

class CommunicationEvent(Event):
    def __init__(self, time: dt.datetime, action_callable, name: str="", id: str=None, satellite: Satellite=None, station: Location=None, comm_pass: ObservationPass=None):
        super().__init__(time, action_callable, name, id)
        self.satellite = satellite
        self.station = station
        self.comm_pass = comm_pass

class Phenomenon(Location):
    def __init__(self, lon_deg: float, lat_deg: float, alt_km: float, start_time: dt.datetime, end_time: dt.datetime, name: str=""):
        super().__init__(lon_deg=lon_deg, lat_deg=lat_deg, alt_km=alt_km, name=name)
        # self.lon_deg = lon_deg
        # self.lat_deg = lat_deg
        # self.alt_km = alt_km
        self.start_time = start_time
        self.end_time = end_time
        # self.name = name
    def __str__(self):
        return "{}: Lon {}, lat {}, alt {}, start {}, end {}".format(self.name, self.lon_deg,self.lat_deg,self.alt_km, self.start_time, self.end_time)
    def __repr__(self):
        return self.__str__()

   
class World():
    def __init__(self, satellites: list=[], constellations: list = [], brokers: list = [], phenomena: list = [], events: list = []):
        self.satellites = satellites
        self.constellations = constellations
        self.brokers = brokers
        self.phenomena = phenomena
        self.events = events
        self.time = dt.datetime.min
        self.history = []
    
    def tick(self, print_forbidden_prefixes=[]):
        if len(self.events):
            _event = self.events.pop(0)

            _print_event_name = True
            for forbidden_names in print_forbidden_prefixes:
                if _event.name.startswith(forbidden_names):
                    _print_event_name = False
                    break
            if _print_event_name:
                print("Executing {}".format(_event))

            outcome = _event.action_callable()
            self.time = _event.time

            self.history.append({
                'time': _event.time,
                'event': _event,
                'states': {
                    'satellites': [copy.deepcopy(s) for s in self.satellites],
                    # Constellations and brokers have a pointer to World, which has a pointer to constellations, which...recursion!
                    # 'constellations': [copy.deepcopy(c) for c in self.constellations],
                    # 'brokers': [copy.deepcopy(b) for b in self.brokers],
                }
            })
            
        else:
            print("No more events")
            
        return len(self.events)

    def add_satellite(self, satellite):
        self.satellite.append(satellite)

    def add_constellation(self, constellation):
        self.constellations.append(constellation)
    
    def add_broker(self, broker):
        self.brokers.append(broker)
    
    def add_event(self, event):
        # print("Adding {}".format(event))
        if (event.time<self.time):
            raise ValueError("Event {} is earlier than sim time {}".format(event, self.time)) 
        bisect.insort(self.events, event, key=lambda x: x.time)
        
    def do_observation(self, observation: ObservationOpportunity, spacecraft: Satellite):
        # Find phenomena close to the observation location in space and at the right time
        # Return a data product and a list of event states
        # print("Obs opp {}".format(observation))
        observed_phenomena = []

        # Now let's see what we observed
        for _phenomenon in self.phenomena:
            if (_phenomenon.start_time <= observation.time and _phenomenon.end_time > observation.time):
                
                # Compute phenomenon location on the planet
                _ph_location_ecf = np.array(pyorbital.astronomy.observer_position(observation.time, _phenomenon.lon_deg, _phenomenon.lat_deg, _phenomenon.alt_km)[0][:3])
                # Compute observation location on the planet
                _obs_location_ecf = np.array(pyorbital.astronomy.observer_position(observation.time, observation.lon_deg, observation.lat_deg, observation.alt_km)[0][:3])
                # Compute SC location
                _sc_location_ecf = np.array(spacecraft.orbit.get_position(observation.time, normalize=False)[0][:3])
                # Compute angle between sc-observation and sc-phenomenon
                _ph_sc_vector = _ph_location_ecf-_sc_location_ecf
                _obs_sc_vector = _obs_location_ecf - _sc_location_ecf
                # print("PhSc {} || ObsSc {}".format(_ph_sc_vector, _obs_sc_vector))
                # print("Dot: {} || N1: {} N2: {}".format(np.dot(_ph_sc_vector,_obs_sc_vector),np.linalg.norm(_ph_sc_vector,2), np.linalg.norm(_obs_sc_vector,2))) 
                # print("Acos: {}, angle: {}".format(np.dot(_ph_sc_vector,_obs_sc_vector)/(np.linalg.norm(_ph_sc_vector,2)*np.linalg.norm(_obs_sc_vector,2)), np.arccos(np.dot(_ph_sc_vector,_obs_sc_vector)/(np.linalg.norm(_ph_sc_vector,2)*np.linalg.norm(_obs_sc_vector,2)))))
                _ph_obs_angle_rad = np.arccos(np.clip(np.dot(_ph_sc_vector,_obs_sc_vector)/(np.linalg.norm(_ph_sc_vector,2)*np.linalg.norm(_obs_sc_vector,2)),-1,1))
                # print("Obs angle (rad) {}".format(_ph_obs_angle_rad))
                # If angle<FOV, return phobservation
                if _ph_obs_angle_rad < spacecraft.instrument_fov_rad[observation.instrument]:
                    # print("Close enough")
                    observed_phenomena.append(_phenomenon)
                else:
                    # print("Too far")
                    pass
        # Store SOMETHING for the completed observation
        spacecraft.data_products.append(observation)
        spacecraft.known_phenomena.append(observed_phenomena)
        
        return True
        







In [9]:
def do_downlink(spacecraft: Satellite, scheduler, comm_pass: ObservationPass): # scheduler is a ConstellationGroundScheduler,defined next 
    # Simple: downlink all. Future: downlink up to x.
    
    # Duration is unused for now
    duration = comm_pass.fall.time - comm_pass.rise.time

    _downlinked = []
    if len(spacecraft.data_products):
        print("Spacecraft {} has {} data products to download".format(spacecraft, len(spacecraft.data_products)))
        
    for data_product in spacecraft.data_products:
        print("  Downlinked {}".format(data_product))
        _downlinked.append(data_product)
        
        matching_requests = scheduler._requests[scheduler._requests['observation']==data_product]
        # print(scheduler._requests)
        # print(matching_requests)
        if len(matching_requests):
            # my_request = matching_requests[0]
            scheduler._requests.loc[scheduler._requests['observation']==data_product, 'status'] = "OK! Data received"
            scheduler._requests.loc[scheduler._requests['observation']==data_product, 'data_product'] = data_product
            scheduler._requests.loc[scheduler._requests['observation']==data_product, 'ready_callback'].item()(data_product)
        

    for _dp in _downlinked:
        spacecraft.data_products.remove(_dp)

In [10]:
class ConstellationGroundScheduler():
    def __init__(self, satellites: list, ground_stations: list, world: World, name="Constellation"):
        self.name = name
        self.satellites = satellites
        self.ground_stations = ground_stations
        self.world = world
        # self.requests = {}
        self._requests = pd.DataFrame(columns=['request', 'satellite', 'observation', 'uplink', 'downlink', 'status', 'data_product', 'scheduled_callback', 'unscheduled_callback', 'ready_callback'])

    def screen_request_for_feasibility(self, _request: ObservationRequest, screen_against_comm_passes:bool=True):
        # Check if a given request conflicts with existing requests.
        # TODO this is horrifyingly expensive because we do not exploit the fact that
        #  requests are sorted. We should improve this, ideally without rebuilding a full on timeline library.
        conflicting_requests = self._requests.loc[
            self._requests.apply(
            lambda x: 
                (x['status']=="Scheduled") and # We have actually scheduled this
                (x['observation'].time+x['observation'].duration > _request.time) and # The end of the other observation is after we start
                (x['observation'].time < _request.time+_request.duration)  # The start of the other observation is before we end
            , axis=1)]
        if len(conflicting_requests):
            return False
        if (screen_against_comm_passes):
            conflicting_uplinks = self._requests.loc[
                self._requests.apply(
                lambda x: 
                    (x['status']=="Scheduled") and # We have actually scheduled this
                    (x['uplink'].fall.time > _request.time) and # The end of the comm pass is after we start
                    (x['uplink'].rise.time < _request.time + _request.duration) # The start of the comm pass is before we end
                , axis=1)]
            if len(conflicting_uplinks):
                return False
            conflicting_downlinks = self._requests.loc[
                self._requests.apply(
                lambda x: 
                    (x['status']=="Scheduled") and # We have actually scheduled this
                    (x['downlink'].fall.time > _request.time) and # The end of the comm pass is after we start
                    (x['downlink'].rise.time < _request.time + _request.duration) # The start of the comm pass is before we end
                , axis=1)]
            if len(conflicting_downlinks):
                return False
        return True
    
    def screen_pass_for_feasibility(self, _obs_pass: ObservationPass, screen_against_comm_passes:bool=False):
        # Check if a given pass conflicts with existing requests.
        # TODO this is horrifyingly expensive because we do not exploit the fact that
        #  requests are sorted. We should improve this, ideally without rebuilding a full on timeline library.
        conflicting_requests = self._requests.loc[
            self._requests.apply(
            lambda x: 
                (x['status']=="Scheduled") and # We have actually scheduled this
                (x['observation'].time+x['observation'].duration > _obs_pass.rise.time) and # The end of the other observation is after we start
                (x['observation'].time < _obs_pass.fall.time)  # The start of the other observation is before we end
            , axis=1)]
        if len(conflicting_requests):
            return False
        if (screen_against_comm_passes):
            conflicting_uplinks = self._requests.loc[
                self._requests.apply(
                lambda x: 
                    (x['status']=="Scheduled") and # We have actually scheduled this
                    (x['uplink'].fall.time > _obs_pass.rise.time) and # The end of the comm pass is after we start
                    (x['uplink'].rise.time < _obs_pass.fall.time) # The start of the comm pass is before we end
                , axis=1)]
            if len(conflicting_uplinks):
                return False
            conflicting_downlinks = self._requests.loc[
                self._requests.apply(
                lambda x: 
                    (x['status']=="Scheduled") and # We have actually scheduled this
                    (x['downlink'].fall.time > _obs_pass.rise.time) and # The end of the comm pass is after we start
                    (x['downlink'].rise.time < _obs_pass.fall.time) # The start of the comm pass is before we end
                , axis=1)]
            if len(conflicting_downlinks):
                return False
        return True

    def schedule_request(
            self,
            request: ObservationRequest,
            current_time: dt.datetime=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
            callback_request_scheduled=lambda req_pass: None,
            callback_request_unscheduled=lambda reason: None,
            callback_request_ready=lambda data_product: None,
            ):
        # Pick the best satellite to fulfill this. This is where we'll need to be smarter. Or not! Just pick something starting the day after.
        print("[{}] Scheduling request {}".format(self.name, request))
        # self.requests[request] = {
        _request_dict = {
            'request': request,
            'satellite': None,
            'observation': None,
            'uplink': None,
            'downlink': None,
            'status': None,
            'data_product': None,
            'scheduled_callback': callback_request_scheduled,
            'unscheduled_callback': callback_request_unscheduled,
            'ready_callback': callback_request_ready,
        }

        _pdrequest = pd.DataFrame([_request_dict])

        self._requests = pd.concat([self._requests, _pdrequest], ignore_index=True)
              
        _opportunities = find_observation_opportunities(
            [request,],
            satellites=self.satellites,
            passes_error_s=60,
            passes_horizon_deg=MIN_HORIZON_ANGLE_FOR_OBS_DEG
        )
        if len(_opportunities):
            # best_request = None
            passes = _opportunities[request]
            # for request, passes in _opportunities.items(): # Only one request, so this just unpacks the opportunities and its OK to reset best_quality below
            if len(passes):
                _best_quality = - np.inf
                _best_satellite = None
                _best_pass = None
                for satellite, satpasses in passes.items():
                    for satpass in satpasses:
                        # Check if the satellite is free at this time.
                        # Query the table of observations for 1. planned, 2. on the satellite we are examining.
                        # Check by time if there is something nearby.
                        # If there is, back off.
                        _pass_is_feasible = self.screen_request_for_feasibility(satpass.highest)
                        if (_pass_is_feasible == False):
                            continue

                        _quality = observation_quality(satpass.highest)
                        if _quality >= _best_quality:
                            _best_quality = _quality
                            _best_satellite = satellite
                            _best_pass = satpass
                        # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))

                print("Best request: {} with {}".format(_best_pass, _best_satellite))
                if (_best_pass is None):
                    print("All observation opportunities are conflicting")
                    self._requests.loc[self._requests['request']==request, 'status'] = "All observation opportunities are conflicting"
                    callback_request_unscheduled("All observation opportunities are conflicting")
                    return -5
            else:
                print("No observation opportunities here")
                print(_opportunities)
                # self.requests[request]['status'] = "No observation opportunities"
                self._requests.loc[self._requests['request']==request, 'status'] = "No observation opportunities"
                callback_request_unscheduled("No observation opportunities")
                return -1
        else:
            print("Something wrong with requests list, did you pass a request?")
        
        # Pick the best contact to tell the satellite. What if the contact is after the request? Whoops.
        _, ul_comm_opportunities = find_contact_opportunities(
            ground_stations=self.ground_stations,
            satellites=self.satellites,
            min_time=current_time,
            max_time=_best_pass.highest.time,
            passes_error_s=60,
            passes_horizon_deg=MIN_HORIZON_ANGLE_FOR_PASS_DEG,
        )

        if ((_best_satellite in ul_comm_opportunities.keys()) and (len(ul_comm_opportunities[_best_satellite])))==0:
            # No contacts!
            print("No timely contact! Maybe we were too greedy")
            # self.requests[request]['status'] = "No timely contact";
            self._requests.loc[self._requests['request']==request, 'status'] = "No timely contact";
            callback_request_unscheduled("No timely contact")
            return -2
        
        earliest_comm_opportunity = None
        earliest_comm_opportunity_station = None
        for comm_opportunity in ul_comm_opportunities[_best_satellite]:
            if self.screen_pass_for_feasibility(comm_opportunity[1], screen_against_comm_passes=False):
                earliest_comm_opportunity = comm_opportunity[1]
                earliest_comm_opportunity_station = comm_opportunity[0]

        if ((earliest_comm_opportunity is None) or (earliest_comm_opportunity_station is None)):
            print("No timely unconflicted contact! Maybe we were too greedy")
            # self.requests[request]['status'] = "No timely contact";
            self._requests.loc[self._requests['request']==request, 'status'] = "No timely unconflicted contact";
            callback_request_unscheduled("No timely unconflicted contact")
            return -2.5

        _best_sat_object = None
        for sat in self.satellites:
            if sat.name == _best_satellite.name:
                _best_sat_object = sat
        if (_best_sat_object is None):
            print("ERROR! Something wrong with finding the satellite")
            # self.requests[request]['status'] = "Could not find best satellite";
            self._requests.loc[self._requests['request']==request, 'status'] = "Could not find best satellite";

            callback_request_unscheduled("Could not find best satellite")
            return -3

        # Let's also schedule a downlink after the event
        # Find downlink opportunities
        _, _dl_comm_opportunities = find_contact_opportunities(
            ground_stations=self.ground_stations,
            satellites=[_best_sat_object, ],
            min_time=_best_pass.highest.time+_best_pass.highest.duration,
            max_time=_best_pass.highest.time+_best_pass.highest.duration+dt.timedelta(hours=48),
            passes_error_s=60,
            passes_horizon_deg=MIN_HORIZON_ANGLE_FOR_PASS_DEG,
        )
        # # Schedule downlink events for those
        if _best_sat_object not in _dl_comm_opportunities.keys() or len(_dl_comm_opportunities[_best_sat_object]) == 0:
            print("  Could not find a suitable downlink")
            # self.requests[request]['status'] = "Could not find best satellite";
            self._requests.loc[self._requests['request']==request, 'status'] = "No timely downlink";
            callback_request_unscheduled("No timely downlink")
            return -4
        
        ## 

        # Passes are sorted by time. An we checked above that there is at least one pass
        _preferred_downlink_pass = _dl_comm_opportunities[_best_sat_object][0]
        _dl_station = _preferred_downlink_pass[0]
        _dl_pass = _preferred_downlink_pass[1]

        _dl_station = None
        _dl_pass = None
        for comm_opportunity in _dl_comm_opportunities[_best_sat_object]:
            if self.screen_pass_for_feasibility(comm_opportunity[1], screen_against_comm_passes=False):
                _dl_pass = comm_opportunity[1]
                _dl_station = comm_opportunity[0]

        if ((_dl_pass is None) or (_dl_station is None)):
            print("  Could not find a suitable unconflicted downlink")
            self._requests.loc[self._requests['request']==request, 'status'] = "No timely unconflicted downlink";
            callback_request_unscheduled("No timely unconflicted downlink")
            return -4.5

        #

        schedule_observation_uplink(self.world, _best_sat_object, earliest_comm_opportunity, _best_pass.highest, earliest_comm_opportunity_station)
        schedule_sat_downlink(_world=self.world, satellite=_best_sat_object, comm_pass = _dl_pass, station = _dl_station, constellation_scheduler=self)
        # # Do downlink
        
        # # Schedule an event where we tell the satellite about this. The event calls schedule_observation
        # - pick the earliest opportunity
        # - check it's early enough (if not return)
        # - schedule an Event at the comm opportunity time that, when triggered, calls schedule_observation with best_request
        self._requests.loc[self._requests['request']==request, 'satellite'] = _best_sat_object
        self._requests.loc[self._requests['request']==request, 'observation'] = _best_pass.highest
        self._requests.loc[self._requests['request']==request, 'uplink'] = earliest_comm_opportunity
        self._requests.loc[self._requests['request']==request, 'downlink'] = _dl_pass
        self._requests.loc[self._requests['request']==request, 'status'] = "Scheduled";

        callback_request_scheduled(_best_pass.highest)
        return 0

    def schedule_downlinks(
        self,
        current_time: dt.datetime=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
        max_time: dt.datetime=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)+dt.timedelta(hours=24)
    ):
        # Find downlink opportunities
        _, comm_opportunities = find_contact_opportunities(
            ground_stations=self.ground_stations,
            satellites=self.satellites,
            min_time=current_time,
            max_time=max_time,
            passes_error_s=60,
            passes_horizon_deg=MIN_HORIZON_ANGLE_FOR_PASS_DEG,
        )
        # Schedule downlink events for those
        for _sat, _comm_passes_and_stations in comm_opportunities.items():
            # print(_comm_pass_and_station)
            # print(_sat)
            for _comm_pass_and_station in _comm_passes_and_stations:
                _station = _comm_pass_and_station[0]
                _comm_pass = _comm_pass_and_station[1]
                schedule_sat_downlink(_world=self.world, satellite=_sat, comm_pass = _comm_pass, station = _station, constellation_scheduler=self)
        # Do downlink
    
    def get_request_status(self, request: ObservationRequest):
        return self._requests[self._requests['request'] == request].status
        if request in self.requests:
            return self.requests[request].status
        else:
            return None

In [11]:
def schedule_observation(_world, satellite: Satellite, obs_opportunity):
    # An observation fires at the time of the observation. It adds known events to the satellite's known_phenomena store.
    # TODO it also adds an observation product to the satellite's 

    # This first bit is quite redundant. What you want is to maintain events for individual agents and then a global copy, right?
    satellite.scheduled_observations.append(obs_opportunity)

    def unlock_satellite(_satellite):
        if _satellite.attitude_controller_state == AttitudeController.INSTRUMENT:
            _satellite.attitude_controller_state = AttitudeController.FREE
            _satellite.busy_with = None
            return True
        else:
            return False

    def lock_satellite_and_observe(_satellite, _obs_opportunity, __world):
        if _satellite.attitude_controller_state != AttitudeController.FREE:
            print("Satellite busy ({})! Sat {} attempted observation {}".format(_satellite.attitude_controller_state, _satellite, _obs_opportunity))
            return False
        _satellite.attitude_controller_state = AttitudeController.INSTRUMENT
        _satellite.busy_with = _obs_opportunity

        _unlock_event = Event(
            name = "Unlock satellite after obs, sat {}".format(_satellite.name),
            time = _obs_opportunity.time+_obs_opportunity.duration,
            action_callable = lambda _sate=satellite: unlock_satellite(_sate)
        )
        _world.add_event(_unlock_event)
        return __world.do_observation(_obs_opportunity, _satellite)
    
    _event = ObservationEvent(
        name = "Obs, sat {}".format(satellite.name),
        time = obs_opportunity.time,
        # Note the kludge of default inputs to make sure the closure works and we capture the variables at the time of creation
        action_callable = lambda _opp=obs_opportunity, _sate=satellite, __world=_world: lock_satellite_and_observe(_sate, _opp, __world),
        satellite=satellite,
        opportunity=obs_opportunity
    )
    _world.add_event(_event)

    return 0

def schedule_observation_uplink(_world: World, satellite: Satellite, comm_opportunity: ObservationPass, obs_opportunity: ObservationOpportunity, station: Location):
    # An observation uplink fires at the time of the uplink. It adds an event that will trigger the observation at the appropriate time. 
    if (comm_opportunity.highest.time>obs_opportunity.time):
        raise ValueError("Uplink {} is after related observation {}".format(comm_opportunity, obs_opportunity))
    
    def unlock_satellite(_satellite, verbose=False):
        if (_satellite.attitude_controller_state == AttitudeController.COMMUNICATION 
            and _satellite.busy_with == comm_opportunity):
            _satellite.attitude_controller_state = AttitudeController.FREE
            _satellite.busy_with = None
            return True
        else:
            if verbose:
                print("Could not unlock satellite after ul comm opportunity with station {} at {}!".format(station, comm_opportunity.highest.time))
            return False
        
    def do_uplink_event(__world: World, __satellite: Satellite, __obsopp: ObservationOpportunity):
        if __satellite.attitude_controller_state == AttitudeController.INSTRUMENT:
            print("Satellite busy! Attempted uplink to sat {} from station {}".format(__satellite, station))
            return False
        __satellite.attitude_controller_state == AttitudeController.COMMUNICATION
        __satellite.busy_with = comm_opportunity
        _unlock_event = Event(
            name = "Unlock uplink, station {} to sat {}".format(station.name, __satellite.name),
            time = comm_opportunity.fall.time,
            action_callable = lambda _sate=satellite: unlock_satellite(_sate)
        )
        __world.add_event(_unlock_event)
        return schedule_observation(__world, __satellite, __obsopp)

    _event = CommunicationEvent(
        name="Uplink, station {} to sat {}".format(station.name, satellite.name),
        time = comm_opportunity.highest.time,
        # action_callable = lambda _w=_world, _s=satellite, _o=obs_opportunity: schedule_observation(_w, _s, _o)
        action_callable = lambda _w=_world, _s=satellite, _o=obs_opportunity: do_uplink_event(_w, _s, _o),
        satellite=satellite,
        station=station,
        comm_pass=comm_opportunity,
    )
    _world.add_event(_event)

def schedule_sat_downlink(
    _world: World,
    satellite: Satellite,
    comm_pass: ObservationPass,
    station: Location,
    constellation_scheduler: ConstellationGroundScheduler
):
    
    def unlock_satellite(_satellite, verbose=False):
        if (_satellite.attitude_controller_state == AttitudeController.COMMUNICATION 
            and _satellite.busy_with == comm_pass):
            _satellite.attitude_controller_state = AttitudeController.FREE
            _satellite.busy_with = None
            return True
        else:
            if verbose:
                print("Could not unlock satellite after dl comm opportunity with station {} at {}!".format(station, comm_pass.highest.time))
            return False
        
    def end_downlink_event(_spacecraft: Satellite, _scheduler: ConstellationGroundScheduler, _comm_pass: ObservationPass):
        do_downlink(spacecraft=_spacecraft, scheduler=_scheduler, comm_pass=_comm_pass)
        return unlock_satellite(_spacecraft)

    def start_downlink_event(_spacecraft: Satellite, _scheduler: ConstellationGroundScheduler, _comm_pass: ObservationPass):
        if _spacecraft.attitude_controller_state == AttitudeController.INSTRUMENT:
            print("Satellite busy! Attempted downlink to sat {} from station {} at ".format(_spacecraft, station, _comm_pass.rise.time))
            return False
        _spacecraft.attitude_controller_state = AttitudeController.COMMUNICATION
        _spacecraft.busy_with = _comm_pass

        _end_event = Event(
            name="End of downlink, station {} from sat {}".format(station.name, satellite.name),
            time = _comm_pass.fall.time,
            # action_callable = lambda _cs=_scheduler, _s=_spacecraft, _c=_comm_pass: do_downlink(spacecraft=_s, scheduler=_cs, comm_pass=_c)
            action_callable = lambda _cs=_scheduler, _s=_spacecraft, _c=_comm_pass: end_downlink_event(_spacecraft=_s, _scheduler=_cs, _comm_pass=_c)
        )
        _world.add_event(_end_event)
        return True


    _event = CommunicationEvent(
        name="Downlink, station {} from sat {}".format(station.name, satellite.name),
        time = comm_pass.rise.time,
        # action_callable = lambda _cs=constellation_scheduler, _s=satellite, _c=comm_pass: do_downlink(spacecraft=_s, scheduler=_cs, comm_pass=_c)
        action_callable = lambda _cs=constellation_scheduler, _s=satellite, _c=comm_pass: start_downlink_event(_spacecraft=_s, _scheduler=_cs, _comm_pass=_c),
        satellite=satellite,
        station=station,
        comm_pass=comm_pass
    )
    _world.add_event(_event)

Something simple. 
- [X] Create a world.
- [X] Add satellites to it.
- [X] Add observation opportunities to the satellites manually
- [X] Click until out of events
- [X] Add a better way to add observation opportunities!
- [X] Add a ConstellationScheduler 

In [12]:
phenomena = [
    Phenomenon(
        lon_deg = -118.,
        lat_deg = 34.,
        alt_km=0.307,
        start_time=min_time,
        end_time=max_time,

    ),
    Phenomenon(8., 45.,  alt_km=0.216, start_time=min_time, end_time=max_time),
    Phenomenon(-80., 34., alt_km=0.041, start_time=min_time, end_time=max_time),
]

In [13]:
phenomena_cities = [
    Phenomenon(
        lon_deg = city.lon_deg,
        lat_deg = city.lat_deg,
        alt_km=city.alt_km,
        start_time=city.min_time,
        end_time=city.max_time,
        name=city.name
    ) for city in one_hundred_sampled_cities
]

In [14]:
one_hundred_sampled_cities[0].min_time

datetime.datetime(2025, 12, 17, 19, 36, 56, 704608)

In [15]:
satellite_agents_simple = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [16]:
world = World(satellites = satellite_agents_simple, phenomena=phenomena)

Manually add observation opportunities

In [17]:
obs_requests = [
    ObservationRequest(-118., 34., min_time=min_time, max_time=max_time, alt_km=0.307),
    ObservationRequest(8., 45., min_time=min_time, max_time=max_time, alt_km=0.216),
    ObservationRequest(-80., 34., min_time=min_time, max_time=max_time, alt_km=0.041),
]

In [18]:
observation_opportunities = find_observation_opportunities(obs_requests, satellites)

In [19]:
best_request = {r: None for r in obs_requests}

for request, passes in observation_opportunities.items():
    _best_quality = - np.inf
    _best_satellite = None
    _best_pass = None
    for satellite, satpasses in passes.items():
        for satpass in satpasses:
            _quality = observation_quality(satpass.highest)
            if _quality >= _best_quality:
                _best_quality = _quality
                _best_satellite = satellite
                _best_pass = satpass
            # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))
    best_request[request] = (_best_satellite, _best_pass)

In [20]:
for req, opp in best_request.items():
    for _sat in world.satellites:
        if _sat.name == opp[0].name:
            _opportunity = opp[1].highest
            print("{} {} {}".format(_sat.name, req, _opportunity))

            schedule_observation(world, _sat, _opportunity)

LOFT YAM-8 Request  | Lon -118.0°, lat 34.0°, alt 0.307 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB Observation  at 2025-12-18 07:22:59.796290 with RGB. Look angle 286.87826626482735 | 35.83454664082302 az/dec deg, zenith angle 168.00268117738023 deg, range 854.8809556481538 km, duration 0:01:00
Ubotica CogniSat-6 HAMMER Request  | Lon 8.0°, lat 45.0°, alt 0.216 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB Observation  at 2025-12-18 02:08:15.875739 with RGB. Look angle 256.044871170689 | 40.86937767277908 az/dec deg, zenith angle 140.4418619010881 deg, range 536.0091425489915 km, duration 0:01:00
Ubotica CogniSat-6 HAMMER Request  | Lon -80.0°, lat 34.0°, alt 0.041 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB Observation  at 2025-12-18 08:12:43.064180 with RGB. Look angle 263.8518606145309 | 32.50217122116019 az/dec deg, zenith angle 140.2826118540574 deg, range 633.2634279108207 km, duration 0:01:0

In [21]:
retcode = 1
while (retcode !=0):
    print("\nTick!")
    print(world.events)
    # print({sat.name: sat.scheduled_observations for sat in world.agents if len(sat.scheduled_observations)})
    retcode = world.tick()


Tick!
[Event Obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 02:08:15.875739, Event Obs, sat LOFT YAM-8 at 2025-12-18 07:22:59.796290, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 08:12:43.064180]
Executing Event Obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 02:08:15.875739

Tick!
[Event Unlock satellite after obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 02:09:15.875739, Event Obs, sat LOFT YAM-8 at 2025-12-18 07:22:59.796290, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 08:12:43.064180]
Executing Event Unlock satellite after obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 02:09:15.875739

Tick!
[Event Obs, sat LOFT YAM-8 at 2025-12-18 07:22:59.796290, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 08:12:43.064180]
Executing Event Obs, sat LOFT YAM-8 at 2025-12-18 07:22:59.796290

Tick!
[Event Unlock satellite after obs, sat LOFT YAM-8 at 2025-12-18 07:23:59.796290, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 08:12:43.064180]
Executing E

In [22]:
[sat.known_phenomena for sat in world.satellites]

[[],
 [],
 [],
 [],
 [[: Lon -118.0, lat 34.0, alt 0.307, start 2025-12-17 19:36:56.704608, end 2025-12-18 19:36:56.704608]],
 [],
 [[: Lon 8.0, lat 45.0, alt 0.216, start 2025-12-17 19:36:56.704608, end 2025-12-18 19:36:56.704608],
  [: Lon -80.0, lat 34.0, alt 0.041, start 2025-12-17 19:36:56.704608, end 2025-12-18 19:36:56.704608]]]

In [23]:
satellite_agents_world = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [24]:
world_with_cities = World(satellites = satellite_agents_world, phenomena=phenomena_cities)



In [25]:
best_request_cities = {r: None for r in one_hundred_sampled_cities}

observation_opportunities_cities = find_observation_opportunities(one_hundred_sampled_cities, satellite_agents_world)

for request, passes in observation_opportunities_cities.items():
    _best_quality = - np.inf
    _best_satellite = None
    _best_pass = None
    for satellite, satpasses in passes.items():
        for satpass in satpasses:
            _quality = observation_quality(satpass.highest)
            if _quality >= _best_quality:
                _best_quality = _quality
                _best_satellite = satellite
                _best_pass = satpass
            # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))
    best_request_cities[request] = (_best_satellite, _best_pass)

In [26]:
for req, opp in best_request_cities.items():
    for _sat in world_with_cities.satellites:
        # Give the event to the right agent
        if _sat.name == opp[0].name:
            # The opportunity is the middle of the pass
            _opportunity = opp[1].highest
            # print("{} {} {}".format(_sat.name, req, _opportunity))

            # This is quite redundant. What you want is to maintain events for individual agents and then a global copy, right?
            _sat.scheduled_observations.append(_opportunity)
            _event = Event(
                name="Observation for {}: {}".format(opp[0].name, opp[1]),
                time = _opportunity.time,
                # The magic is here: we add an event that does an observation and stores the result in the agent's known_phenomena bin
                # Note the kludge of default inputs to make sure the closure works and we capture the variables at the time of creation
                # action_callable = lambda _opp=_opportunity, _satname=_sat.name: print("{} with {}".format(_opp, _satname)) #_sat.known_phenomena.append(world.do_observation(_opportunity, _sat))
                action_callable = lambda _opp=_opportunity, _sate=_sat: world_with_cities.do_observation(_opp, _sate)
            )
            world_with_cities.add_event(_event)

In [27]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    # print(world_cities.events)
    # print({sat.name: sat.scheduled_observations for sat in world.agents if len(sat.scheduled_observations)})
    retcode = world_with_cities.tick()

Executing Event Observation for Ubotica CogniSat-6 HAMMER: Pass start: 2025-12-17 19:47:55.586547, highest: 2025-12-17 19:52:32.873357, fall: 2025-12-17 19:57:05.470163 at 2025-12-17 19:52:32.873357
Executing Event Observation for LOFT YAM-5: Pass start: 2025-12-17 19:47:40.763658, highest: 2025-12-17 19:53:03.875040, fall: 2025-12-17 19:58:29.982965 at 2025-12-17 19:53:03.875040
Executing Event Observation for Ubotica CogniSat-6 HAMMER: Pass start: 2025-12-17 19:52:43.590449, highest: 2025-12-17 19:57:19.145453, fall: 2025-12-17 20:02:00.699128 at 2025-12-17 19:57:19.145453
Executing Event Observation for Ubotica CogniSat-6 HAMMER: Pass start: 2025-12-17 19:52:48.409123, highest: 2025-12-17 19:57:27.955638, fall: 2025-12-17 20:02:07.501741 at 2025-12-17 19:57:27.955638
Executing Event Observation for Ubotica CogniSat-6 HAMMER: Pass start: 2025-12-17 19:56:30.406949, highest: 2025-12-17 20:00:55.427479, fall: 2025-12-17 20:05:20.908590 at 2025-12-17 20:00:55.427479
Executing Event Obse

In [28]:
satellite_agents_sched = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

world_with_scheduler = World(satellites = satellite_agents_sched, phenomena=phenomena)

scheduler = ConstellationGroundScheduler(satellites=satellite_agents_sched, ground_stations=ground_stations, world=world_with_scheduler)

world_with_scheduler.add_constellation(scheduler)

In [29]:
for request in obs_requests:
    scheduler.schedule_request(
        request=request,
        current_time=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
        callback_request_scheduled=lambda obs: print("Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("Request {} ready with DP {}!".format(request, dp)),
        )

[Constellation] Scheduling request Request  | Lon -118.0°, lat 34.0°, alt 0.307 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB
Best request: Pass start: 2025-12-18 07:20:15.304105, highest: 2025-12-18 07:22:59.796290, fall: 2025-12-18 07:25:45.008209 with LOFT YAM-8
Request Request  | Lon -118.0°, lat 34.0°, alt 0.307 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB scheduled for observation Observation  at 2025-12-18 07:22:59.796290 with RGB. Look angle 286.87826626482735 | 35.83454664082302 az/dec deg, zenith angle 168.00268117738023 deg, range 854.8809556481538 km, duration 0:01:00!
[Constellation] Scheduling request Request  | Lon 8.0°, lat 45.0°, alt 0.216 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB
Best request: Pass start: 2025-12-18 02:06:14.598605, highest: 2025-12-18 02:08:15.875739, fall: 2025-12-18 02:10:25.553094 with Ubotica CogniSat-6 HAMMER
Request Request  | Lon 8.0°, lat 45.0°, alt 0.216

In [30]:
scheduler.schedule_downlinks()

In [31]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    retcode = world_with_scheduler.tick(print_forbidden_prefixes=["Downlink", "End of downlink"])
    # print(world_with_scheduler.events)

Executing Event Uplink, station KSAT Svalbard to sat Ubotica CogniSat-6 HAMMER at 2025-12-18 00:45:09.716044
Executing Event Unlock uplink, station KSAT Svalbard to sat Ubotica CogniSat-6 HAMMER at 2025-12-18 00:47:11.148030
Executing Event Obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 02:08:15.875739
Executing Event Unlock satellite after obs, sat Ubotica CogniSat-6 HAMMER at 2025-12-18 02:09:15.875739
Spacecraft Ubotica CogniSat-6 HAMMER has 1 data products to download
  Downlinked Observation  at 2025-12-18 02:08:15.875739 with RGB. Look angle 256.044871170689 | 40.86937767277908 az/dec deg, zenith angle 140.4418619010881 deg, range 536.0091425489915 km, duration 0:01:00
Request Request  | Lon -80.0°, lat 34.0°, alt 0.041 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB ready with DP Observation  at 2025-12-18 02:08:15.875739 with RGB. Look angle 256.044871170689 | 40.86937767277908 az/dec deg, zenith angle 140.4418619010881 deg, range 536.0091425489915 k

In [32]:
# scheduler.requests

In [33]:
scheduler._requests

,request,satellite,observation,uplink,downlink,status,data_product,scheduled_callback,unscheduled_callback,ready_callback
0,"Request | Lon -118.0°, lat 34.0°, alt 0.307 k...",LOFT YAM-8,Observation at 2025-12-18 07:22:59.796290 wit...,"Pass start: 2025-12-18 07:03:01.423364, highes...","Pass start: 2025-12-20 06:43:43.973353, highes...",OK! Data received,Observation at 2025-12-18 07:22:59.796290 wit...,<function <lambda> at 0x12a433560>,<function ConstellationGroundScheduler.<lambda...,<function <lambda> at 0x12a4336a0>
1,"Request | Lon 8.0°, lat 45.0°, alt 0.216 km f...",Ubotica CogniSat-6 HAMMER,Observation at 2025-12-18 02:08:15.875739 wit...,"Pass start: 2025-12-18 00:43:06.288187, highes...","Pass start: 2025-12-20 01:40:12.534339, highes...",OK! Data received,Observation at 2025-12-18 02:08:15.875739 wit...,<function <lambda> at 0x12a5d4680>,<function ConstellationGroundScheduler.<lambda...,<function <lambda> at 0x12a5d4720>
2,"Request | Lon -80.0°, lat 34.0°, alt 0.041 km...",Ubotica CogniSat-6 HAMMER,Observation at 2025-12-18 08:12:43.064180 wit...,"Pass start: 2025-12-18 08:04:11.138383, highes...","Pass start: 2025-12-20 07:30:37.419572, highes...",OK! Data received,Observation at 2025-12-18 08:12:43.064180 wit...,<function <lambda> at 0x12a5e8040>,<function ConstellationGroundScheduler.<lambda...,<function <lambda> at 0x12a5e80e0>


Where do we go from here?

The obvious: add conflicts between observations. For now, we can stay in discrete land where an opportunity is an instantaneous thing. Or we can create multiple copies all along the path.
Conflict: two opportunities are on the same satellite and closer than some amount of time. The time should account for (i) some fixed setup (think "wait for the mirror to stop flapping") plus some time related to reorientation.
This misses the fact that one could point the spacecraft slightly away from the target and still get it. 
Can we get that in pre-processing?

Solve the one-off scheduling problem with constraints (greedy, find the best observation that is feasible).

Solve the one-off scheduling problem with constraints including comms (greedy, find the best observation that is feasible after we can talk to a given satellite).

Solve the batch scheduling problem with constraints (ILP? For old times' sake).

Multiple instruments. Spacecraft should have an instrument attached. Requests and S/C have an instrument.

Follow-on requests. A detection causes a follow-up request. Simulate.

Write up the three cases of interest:
- Submit a request and you immediately hear back. Unsubmitting requests is free.
    - Use your favorite black-box scheduling algorithm. Submitting a request==evaluating a constraint.
- Submit a request and you immediately hear back. Unsubmitting requests is expensive.
    - Use your favorite non-backtracking scheduling algorithm. Once you choose, no regrets.
- Submit a request and you don't hear back. This is a DMU problem.
    - State:
    - Actions: schedule an observation on a satellite.
    - Transitions: from "unscheduled" to "scheduled" to "executed" or "rejected" for every observation. 
    - Observations: when a file is downloaded, we find out.
    - Rewards: k if we get an observation, 0 otherwise.

- [ ] Constellation: 
  - Input:
      - [X] A new request
      - [X] A schedule of observations assigned to agents
      - [X] (can compute) Alternate windows for the assigned observations
  - Output:
      - A new schedule of observations assigned to agents
      - [X] A bool indicating whether the request was assigned
  - Formulate the optimization problem:
      - Identify which observations can be unscheduled (are not yet committed)
      - Formulate the profit-maximizing problem of assigning everything
      - Solve the problem
      - Check if the new observation is in.
- [ ] Broker
  - Input:
      - A new request
  - Output:
      - Time when the request is scheduled
  - List all windows across all constellations
  - Pick the best one
  - Send the request
- API:
    - [ ] Constellation
        - Input:
            - [X] Submit a request
            - [X] Retrieve a request status
        - Output:
            - [X] Report data acquired
            - [X] Report event
            - [X] Report unscheduled task
    - [ ] Broker
        - Input:
            - Submit a workflow request
            - Status update on observation request
        - Output
            -  Observation requests
            -  Status updates on workflow requests

In [34]:
# Let's have multiple constellations!

# Behind the scenes, this pulls from Celestrak if we do not specify tle_file

satellites_LOFT = [
    # Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    # Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

satellites_ubotica = [
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    ]

satellites_aerospace = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   

]

satellites_multischedulers = satellites_LOFT+satellites_ubotica+satellites_aerospace

In [35]:
world_with_schedulers = World(satellites = satellites_multischedulers, phenomena=phenomena_cities)

scheduler_LOFT = ConstellationGroundScheduler(satellites=satellites_LOFT, ground_stations=ground_stations, world=world_with_schedulers, name="LOFT")
scheduler_ubotica = ConstellationGroundScheduler(satellites=satellites_ubotica, ground_stations=ground_stations, world=world_with_schedulers, name="UBOTICA")
scheduler_aerospace = ConstellationGroundScheduler(satellites=satellites_aerospace, ground_stations=ground_stations, world=world_with_schedulers, name="AC")

world_with_schedulers.add_constellation(scheduler_LOFT)
world_with_schedulers.add_constellation(scheduler_ubotica)
world_with_schedulers.add_constellation(scheduler_aerospace)

In [36]:
sampled_world_cities_LOFT = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=0)
sampled_world_cities_ubotica = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=1)
sampled_world_cities_aerospace = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=2)

obs_request_LOFT = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_LOFT.iterrows()
]

obs_request_ubotica = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_ubotica.iterrows()
]

obs_request_aerospace = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_aerospace.iterrows()
]

In [37]:
sim_start_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

for request in obs_request_LOFT:
    scheduler_LOFT.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_ubotica:
    scheduler_ubotica.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_aerospace:
    scheduler_aerospace.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
        )

[LOFT] Scheduling request Request Korla | Lon 86.1746°, lat 41.7259°, alt 0.307 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB
Best request: Pass start: 2025-12-18 18:32:45.529681, highest: 2025-12-18 18:36:01.969305, fall: 2025-12-18 18:39:23.567292 with LOFT YAM-10
[LOFT] Request Request Korla | Lon 86.1746°, lat 41.7259°, alt 0.307 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB scheduled for observation Observation  at 2025-12-18 18:36:01.969305 with RGB. Look angle 75.78471807320632 | 55.15375053352562 az/dec deg, zenith angle 161.01343050939184 deg, range 712.6799490620754 km, duration 0:01:00!
[LOFT] Scheduling request Request Đà Lạt | Lon 108.4383°, lat 11.9417°, alt 0.307 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB
Best request: Pass start: 2025-12-18 07:56:12.688837, highest: 2025-12-18 07:57:43.756604, fall: 2025-12-18 07:59:12.754801 with LOFT YAM-6
[LOFT] Request Request Đà Lạt | Lon 108.438

In [38]:
# scheduler_LOFT.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_ubotica.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_aerospace.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))

In [39]:
world_with_schedulers.events

[Event Uplink, station KSAT Punta Arenas to sat LOFT YAM-6 at 2025-12-17 19:43:33.284779,
 Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242,
 Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242,
 Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242,
 Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242,
 Event Uplink, station KSAT Singapore to sat Ubotica CogniSat-6 HAMMER at 2025-12-17 19:50:12.223878,
 Event Uplink, station KSAT Singapore to sat Ubotica CogniSat-6 HAMMER at 2025-12-17 19:50:12.223878,
 Event Uplink, station KSAT Singapore to sat Ubotica CogniSat-6 HAMMER at 2025-12-17 19:50:12.223878,
 Event Uplink, station KSAT Singapore to sat Ubotica CogniSat-6 HAMMER at 2025-12-17 19:50:12.223878,
 Event Uplink, station KSAT Singapore to sat Ubotica CogniSat-6 HAMMER at 2025-12-17 19:50:12.223878,
 Event Uplink, station KSAT Nuuk to sat Ubotica CogniSat-6 H

In [40]:
retcode = 1
while (retcode !=0):
    retcode = world_with_schedulers.tick()
    # retcode = world_with_schedulers.tick(print_forbidden_prefixes=["Downlink", "End of downlink", "Unlock uplink", "Unlock satellite after obs"])

Executing Event Uplink, station KSAT Punta Arenas to sat LOFT YAM-6 at 2025-12-17 19:43:33.284779
Executing Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242
Executing Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242
Executing Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242
Executing Event Uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:43:35.672242
Executing Event Unlock uplink, station KSAT Punta Arenas to sat LOFT YAM-6 at 2025-12-17 19:44:11.283001
Executing Event Unlock uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:45:10.483266
Executing Event Unlock uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:45:10.483266
Executing Event Unlock uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:45:10.483266
Executing Event Unlock uplink, station KSAT Vardo to sat AEROCUBE 18B at 2025-12-17 19:45:10.483266
Executing Event Uplink, s

In [41]:
def request_statistics(requests_pd):
    total_requests = len(requests_pd)
    all_statuses = set(requests_pd.status.values)    
    for s in all_statuses:
        # matching_statuses = sum([1 if (r['status']==s) else 0 for r in requests.values()])
        matching_statuses = len(requests_pd[requests_pd.status==s])
        print("{}/{} ({}%) of requests are in status {}".format(matching_statuses,total_requests, matching_statuses/total_requests*100, s))

In [42]:
print("LOFT")
request_statistics(scheduler_LOFT._requests)

print("Ubotica")
request_statistics(scheduler_ubotica._requests)

print("AC")
request_statistics(scheduler_aerospace._requests)




LOFT
47/50 (94.0%) of requests are in status OK! Data received
3/50 (6.0%) of requests are in status No timely contact
Ubotica
43/50 (86.0%) of requests are in status OK! Data received
7/50 (14.000000000000002%) of requests are in status All observation opportunities are conflicting
AC
50/50 (100.0%) of requests are in status OK! Data received


In [ ]:
class Broker():
    def __init__(self, constellations: list[ConstellationGroundScheduler], world: World, name="Broker"):
        self.name = name
        self.constellations = constellations
        # self.known_satellites = known_satellites
        self.world = world
        self._requests = pd.DataFrame(columns=['request', 'requested_pass', 'requested_constellation', 'requested_satellite', 'constellation', 'satellite', 'assigned_pass', 'assigned_downlink', 'status', 'data_product', 'scheduled_callback', 'unscheduled_callback', 'ready_callback'])

        # self.requests = {}

    # Broadly, look at the ephemerides, find the best option, find the corresponding constellation, give them a window around that.
    def schedule_request(
            self,
            request: ObservationRequest,
            current_time: dt.datetime=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
            number_of_submissions: int=1,
            ):
        # Pick the best satellite to fulfill this. This is where we'll need to be smarter. Or not! Just pick something starting the day after.
        print("[{}] scheduling request {}".format(self.name, request))
        # self.requests[request] = {


        _known_satellites = []
        _known_satellites_by_constellation = {}
        for constellation in self.constellations:
            _known_satellites += constellation.satellites
            for _sat in constellation.satellites:
                _known_satellites_by_constellation[_sat] = constellation

        _opportunities = find_observation_opportunities(
            [request,],
            satellites=_known_satellites,
            passes_error_s=60,
            passes_horizon_deg=MIN_HORIZON_ANGLE_FOR_OBS_DEG
        )
        # self.requests[request]['opportunities'] = _opportunities
        # self._requests.loc[self._requests['request']==request, 'opportunities'] = _opportunities

        passes = _opportunities[request]

        if len(passes):
            
            # TODO sort passes with key observation_quality(passes[s][p_index].highest)
            sorted_passes = [(satellite, satpass) for satellite, satpasses in passes.items() for satpass in satpasses]
            # Sort by quality
            sorted_passes.sort(key=lambda x: observation_quality(x[1].highest), reverse=True)
            # print("Sorted passes: ", sorted_passes)
            
            _best_satellite = sorted_passes[0][0]
            _best_pass = sorted_passes[0][1]
            _best_quality = observation_quality(_best_pass.highest)

            # for satellite, satpasses in passes.items():
            #     for satpass in satpasses:
            #         _quality = observation_quality(satpass.highest)
            #         if _quality > _best_quality:
            #             assert False, "How did I find a better pass?"
            #             _best_quality = _quality
            #             _best_satellite = satellite
            #             _best_pass = satpass
            #         # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))
            # # best_request = (_best_satellite, _best_pass)
            # assert _best_satellite==sorted_passes[0][0], "ERROR: disagreement on best satellite"
            # assert _best_pass==sorted_passes[0][1], "ERROR: disagreement on best pass"

            print("Best request: {} with {}".format(_best_pass, _best_satellite))
        else:
            print("No observation opportunities here")
            # self.requests[request]['status'] = "No observation opportunities";
            _request_dict = {
                'request': request,
                'requested_pass' : None,
                'requested_constellation' : None,
                'requested_satellite' : None,
                'constellation': None,
                'satellite': None,
                'assigned_pass': None,
                'assigned_downlink': None,
                'status': "No observation opportunities",
                'data_product': None,
                'scheduled_callback': lambda x: None,
                'unscheduled_callback': lambda x: None,
                'ready_callback': lambda x: None,
            }
            _pdrequest = pd.DataFrame([_request_dict])
            self._requests = pd.concat([self._requests, _pdrequest], ignore_index=True)

            # self._requests.loc[self._requests['request']==request, 'status'] = "No observation opportunities"
            return -1

        for opportunity_ix in range(min(number_of_submissions, len(sorted_passes))):
            _best_pass = sorted_passes[opportunity_ix][1]
            _best_satellite = sorted_passes[opportunity_ix][0]

            if (_best_pass is not None) and (_best_satellite is not None):
                _best_constellation = _known_satellites_by_constellation[_best_satellite]

                def callback_request_scheduled(assigned_pass):
                    print(" [{}] confirmed scheduling of request {} from {}".format(self.name, request, _best_constellation))
                    # TODO also filter by best_pass.
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'assigned_pass'] = assigned_pass
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'status'] = "Scheduled"
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'constellation'] = _best_constellation
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'satellite'] = _best_satellite
                    return
                
                def callback_request_unscheduled(reason):
                    print(" [{}] received UNscheduling of request {} from {}".format(self.name, request, _best_constellation))
                    # TODO also filter by best_pass.
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'assigned_pass'] = None
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'status'] = reason
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'constellation'] = None
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'satellite'] = None
                    return
                
                def callback_request_ready(data_product):
                    print(" [{}: ] data ready for request {} from {}".format(self.name, request, _best_constellation))
                    # self._requests.loc[self._requests['request']==request, 'assigned_pass'] = None
                    # TODO also filter by best_pass.
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'status'] = "Completed"
                    # self._requests.loc[self._requests['request']==request, 'constellation'] = None
                    # self._requests.loc[self._requests['request']==request, 'satellite'] = None
                    self._requests.loc[((self._requests['request']==request) & (self._requests['requested_pass']==_best_pass)), 'data_product'] = data_product
                    return

                _constellation_request = ObservationRequest(
                    lon_deg=request.lon_deg,
                    lat_deg=request.lat_deg,
                    min_time=_best_pass.rise.time-dt.timedelta(minutes=1), # This is the magic, we constrain the request to the constellation AND TIME that we like.
                    max_time=_best_pass.fall.time+dt.timedelta(minutes=1),
                    alt_km=request.alt_km,
                    instrument=request.instrument,
                    request_name=request.name,
                )

                _request_dict = {
                    'request': request,
                    'requested_pass' : _best_pass,
                    'requested_constellation' : _best_constellation,
                    'requested_satellite' :_best_satellite,
                    'constellation': None,
                    'satellite': None,
                    'assigned_pass': None,
                    'assigned_downlink': None,
                    'status': "Submitted",
                    'data_product': None,
                    'scheduled_callback': lambda x: None,
                    'unscheduled_callback': lambda x: None,
                    'ready_callback': lambda x: None,
                }
                _pdrequest = pd.DataFrame([_request_dict])
                self._requests = pd.concat([self._requests, _pdrequest], ignore_index=True)

                # Submit the request to the relevant constellation
                _best_constellation.schedule_request(
                    request=_constellation_request,
                    current_time=current_time,
                    callback_request_scheduled=callback_request_scheduled,
                    callback_request_unscheduled=callback_request_unscheduled,
                    callback_request_ready=callback_request_ready,
                )
                


    # Do the silly thing: decompose the workflow deterministically, then assign ALL those requests...
    def schedule_workflow(
            self,
            workflow,
            current_time: dt.datetime=dt.datetime.now(dt.timezone.utc).replace(tzinfo=None),
    ):
        pass





In [115]:
sim_start_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

satellites_LOFT = [
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
]

satellites_ubotica = [
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    ]

satellites_aerospace = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   
]

satellites_multischedulers = satellites_LOFT+satellites_ubotica+satellites_aerospace

In [116]:
world_with_brokers = World(satellites = satellites_multischedulers, phenomena=phenomena_cities)

N_BACKGROUND_SAMPLES = 0

scheduler_LOFT = ConstellationGroundScheduler(satellites=satellites_LOFT, ground_stations=ground_stations, world=world_with_brokers, name="LOFT")
scheduler_ubotica = ConstellationGroundScheduler(satellites=satellites_ubotica, ground_stations=ground_stations, world=world_with_brokers, name="UBOTICA")
scheduler_aerospace = ConstellationGroundScheduler(satellites=satellites_aerospace, ground_stations=ground_stations, world=world_with_brokers, name="AC")

world_with_brokers.add_constellation(scheduler_LOFT)
world_with_brokers.add_constellation(scheduler_ubotica)
world_with_brokers.add_constellation(scheduler_aerospace)

sampled_world_cities_LOFT = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES,weights='population',axis=0, random_state=0)
sampled_world_cities_ubotica = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES,weights='population',axis=0, random_state=1)
sampled_world_cities_aerospace = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES,weights='population',axis=0, random_state=2)
obs_request_LOFT = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_LOFT.iterrows()
]
obs_request_ubotica = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_ubotica.iterrows()
]
obs_request_aerospace = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_aerospace.iterrows()
]


for request in obs_request_LOFT:
    scheduler_LOFT.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_ubotica:
    scheduler_ubotica.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_aerospace:
    scheduler_aerospace.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
        )
    
# scheduler_LOFT.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_ubotica.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_aerospace.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))

In [117]:
broker = Broker(constellations=[scheduler_LOFT, scheduler_ubotica, scheduler_aerospace], world=world_with_brokers)

world_with_brokers.add_broker(broker)

In [118]:
sampled_world_cities_broker = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=3)

number_of_submissions = 3

obs_requests_broker = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_broker.iterrows()
]

for _req in obs_requests_broker:
    broker.schedule_request(_req, number_of_submissions=number_of_submissions)

[Broker] scheduling request Request Kaduna | Lon 7.4333°, lat 10.5167°, alt 0.307 km from 2025-12-17 19:36:56.704608 to 2025-12-18 19:36:56.704608 with RGB
Sorted passes:  [(Ubotica CogniSat-6 HAMMER, Pass start: 2025-12-18 01:57:48.495494, highest: 2025-12-18 01:59:52.703176, fall: 2025-12-18 02:01:52.750226), (AEROCUBE 18B, Pass start: 2025-12-17 21:31:42.534577, highest: 2025-12-17 21:34:11.578807, fall: 2025-12-17 21:36:44.528583), (LOFT YAM-10, Pass start: 2025-12-18 00:42:33.962308, highest: 2025-12-18 00:45:52.597114, fall: 2025-12-18 00:49:08.485434), (AEROCUBE 18A, Pass start: 2025-12-17 22:38:38.444834, highest: 2025-12-17 22:40:10.606855, fall: 2025-12-17 22:41:44.652868), (LOFT YAM-6, Pass start: 2025-12-18 02:34:02.413058, highest: 2025-12-18 02:34:57.833228, fall: 2025-12-18 02:35:53.253398), (Ubotica CogniSat-6 HAMMER, Pass start: 2025-12-18 14:52:35.710388, highest: 2025-12-18 14:54:03.557671, fall: 2025-12-18 14:55:34.109513), (LOFT YAM-6, Pass start: 2025-12-18 14:11:

In [119]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    retcode = world_with_brokers.tick(print_forbidden_prefixes=["Downlink", "End of downlink", "Unlock uplink", "Unlock satellite after obs"])
    # print(world_with_scheduler.events)

Executing Event Uplink, station KSAT Troll to sat LOFT YAM-6 at 2025-12-18 02:13:59.075817
Executing Event Obs, sat LOFT YAM-6 at 2025-12-18 02:34:05.059563
Executing Event Uplink, station KSAT Nuuk to sat AEROCUBE 18B at 2025-12-18 03:37:52.443869
Executing Event Obs, sat AEROCUBE 18B at 2025-12-18 03:54:19.008380
Executing Event Uplink, station KSAT Nuuk to sat AEROCUBE 18A at 2025-12-18 04:43:28.781856
Executing Event Uplink, station KSAT Nuuk to sat AEROCUBE 18A at 2025-12-18 04:43:28.781856
Executing Event Uplink, station KSAT Nuuk to sat AEROCUBE 18A at 2025-12-18 04:43:28.781856
Executing Event Uplink, station KSAT Nuuk to sat AEROCUBE 18A at 2025-12-18 04:43:28.781856
Executing Event Obs, sat AEROCUBE 18A at 2025-12-18 04:51:31.406483
Executing Event Obs, sat AEROCUBE 18A at 2025-12-18 04:51:31.406483
Executing Event Obs, sat AEROCUBE 18A at 2025-12-18 04:57:17.839411
Executing Event Obs, sat AEROCUBE 18A at 2025-12-18 04:57:17.839411
Executing Event Uplink, station KSAT Svalba

In [120]:
request_statistics(broker._requests)

35/151 (23.178807947019866%) of requests are in status No timely contact
0/151 (0.0%) of requests are in status nan
1/151 (0.6622516556291391%) of requests are in status No observation opportunities
36/151 (23.841059602649008%) of requests are in status Scheduled
46/151 (30.4635761589404%) of requests are in status All observation opportunities are conflicting
32/151 (21.192052980132452%) of requests are in status Completed


In [121]:
def retell_history(world: World):
    for _chronicle in world.history:
        print("Time: {}. Event: {}".format(_chronicle['time'], _chronicle['event']))
        if type(_chronicle['event'])==ObservationEvent:
            print("Observation: sat {} and opportunity {}".format(_chronicle['event'].satellite, _chronicle['event'].opportunity))
        if type(_chronicle['event'])==CommunicationEvent:
            print("Communication: station {} to sat {} during pass {}".format(_chronicle['event'].station, _chronicle['event'].satellite, _chronicle['event'].comm_pass))

In [122]:
retell_history(world_with_brokers)

Time: 2025-12-18 02:13:59.075817. Event: Event Uplink, station KSAT Troll to sat LOFT YAM-6 at 2025-12-18 02:13:59.075817
Communication: station Location KSAT Troll | Lon 2.53219°, lat -72.01243°, alt 0 km to sat LOFT YAM-6 during pass Pass start: 2025-12-18 02:11:43.341037, highest: 2025-12-18 02:13:59.075817, fall: 2025-12-18 02:16:11.886990
Time: 2025-12-18 02:16:11.886990. Event: Event Unlock uplink, station KSAT Troll to sat LOFT YAM-6 at 2025-12-18 02:16:11.886990
Time: 2025-12-18 02:34:05.059563. Event: Event Obs, sat LOFT YAM-6 at 2025-12-18 02:34:05.059563
Observation: sat LOFT YAM-6 and opportunity Observation  at 2025-12-18 02:34:05.059563 with RGB. Look angle 258.2839897411559 | 32.921929308606934 az/dec deg, zenith angle 135.57021326154998 deg, range 806.6893633796532 km, duration 0:01:00
Time: 2025-12-18 02:35:05.059563. Event: Event Unlock satellite after obs, sat LOFT YAM-6 at 2025-12-18 02:35:05.059563
Time: 2025-12-18 03:37:52.443869. Event: Event Uplink, station KSAT

In [123]:
broker._requests

,request,requested_pass,requested_constellation,requested_satellite,constellation,satellite,assigned_pass,assigned_downlink,status,data_product,scheduled_callback,unscheduled_callback,ready_callback,opportunities
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,{LOFT YAM-6: [Pass start: 2025-12-18 02:34:02....
1,"Request Kaduna | Lon 7.4333°, lat 10.5167°, al...","Pass start: 2025-12-18 01:57:48.495494, highes...",<__main__.ConstellationGroundScheduler object ...,Ubotica CogniSat-6 HAMMER,None,None,None,None,No timely contact,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN
2,"Request Kaduna | Lon 7.4333°, lat 10.5167°, al...","Pass start: 2025-12-17 21:31:42.534577, highes...",<__main__.ConstellationGroundScheduler object ...,AEROCUBE 18B,None,None,None,None,No timely contact,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN
3,"Request Kaduna | Lon 7.4333°, lat 10.5167°, al...","Pass start: 2025-12-18 00:42:33.962308, highes...",<__main__.ConstellationGroundScheduler object ...,LOFT YAM-10,None,None,None,None,No timely contact,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN
4,"Request Wayaobu | Lon 109.6752°, lat 37.1427°,...","Pass start: 2025-12-18 18:54:12.848704, highes...",<__main__.ConstellationGroundScheduler object ...,Ubotica CogniSat-6 HAMMER,<__main__.ConstellationGroundScheduler object ...,Ubotica CogniSat-6 HAMMER,Observation at 2025-12-18 18:56:17.831876 wit...,None,Scheduled,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,"Request Yulin | Lon 110.181°, lat 22.654°, alt...","Pass start: 2025-12-18 14:49:52.031908, highes...",<__main__.ConstellationGroundScheduler object ...,AEROCUBE 18B,None,None,None,None,All observation opportunities are conflicting,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN
147,"Request Yulin | Lon 110.181°, lat 22.654°, alt...","Pass start: 2025-12-18 16:52:13.860307, highes...",<__main__.ConstellationGroundScheduler object ...,LOFT YAM-10,None,None,None,None,All observation opportunities are conflicting,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN
148,"Request Dalian | Lon 121.6°, lat 38.9°, alt 0....","Pass start: 2025-12-18 14:17:23.343656, highes...",<__main__.ConstellationGroundScheduler object ...,AEROCUBE 18A,None,None,None,None,All observation opportunities are conflicting,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN
149,"Request Dalian | Lon 121.6°, lat 38.9°, alt 0....","Pass start: 2025-12-18 18:19:19.179307, highes...",<__main__.ConstellationGroundScheduler object ...,LOFT YAM-6,None,None,None,None,All observation opportunities are conflicting,None,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,<function Broker.schedule_request.<locals>.<la...,NaN


In [110]:
def plot_event(_chronicle: dict, world: World, ax=None):
    if ax is None:
        figglobal = plt.figure(figsize=(10,5))
        ax = figglobal.add_subplot(1,1,1, projection=ccrs.Robinson())
        ax.set_global()
        ax.coastlines()

    _time_to_plot_ground_track = dt.timedelta(seconds=15*60)
    _dt_to_plot_ground_track = dt.timedelta(seconds=60)
    time_steps_for_plotting = [_chronicle['time']- _dt_to_plot_ground_track*i for i in range(int(math.ceil(_time_to_plot_ground_track/_dt_to_plot_ground_track)))]

    ax.text(0,0,"{}".format(_chronicle['time']), transform=ax.transAxes)

    # for satellite in world.satellites:
    #     _orbit = satellite.orbit
    #     _llas = [_orbit.get_lonlatalt(t) for t in time_steps_for_plotting]
    #     # ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic())
    #     ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic())
    
    constellation_palette = cmap['viridis'].resampled(len(world.constellations))

    for constellation_ix, constellation in enumerate(world.constellations):
        constellation_color = constellation_palette(constellation_ix/len(world.constellations))
        for ground_station in constellation.ground_stations:
            ax.plot(float(ground_station.lon_deg), float(ground_station.lat_deg), '*', transform=ccrs.PlateCarree(), color=constellation_color)

        for satellite in constellation.satellites:
            _orbit = satellite.orbit
            _llas = [_orbit.get_lonlatalt(t) for t in time_steps_for_plotting]
            # ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic())
            ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic(), color=constellation_color)
    
    if type(_chronicle['event'])==ObservationEvent:
        ax.plot(
            [_chronicle['event'].opportunity.lon_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0]],
            [_chronicle['event'].opportunity.lat_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1]],
            ':k',
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].opportunity.lon_deg,
            _chronicle['event'].opportunity.lat_deg,
            'Dr',
            markersize=10,
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0],
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1],
            '.',
            markersize=10,
            transform=ccrs.Geodetic()
        )

    elif type(_chronicle['event'])==CommunicationEvent:
        ax.plot(
            [_chronicle['event'].station.lon_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0]],
            [_chronicle['event'].station.lat_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1]],
            '-.k',
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].station.lon_deg,
            _chronicle['event'].station.lat_deg,
            '*',
            markersize=10,
            transform=ccrs.Geodetic()
        )
        ax.plot(
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0],
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1],
            '.',
            markersize=10,
            transform=ccrs.Geodetic()
        )
    return ax


def plot_history(world: World):
    artists = []
    for _chronicle_ix, _chronicle in enumerate(world.history):
        _ax = plot_event(_chronicle, world)
        plt.savefig("FRAME_{:05d}.png".format(_chronicle_ix))
        # artists.append(_ax)
        
    # plt.show()
        # if type(_chronicle['event'])==ObservationEvent:
        #     print("Observation: sat {} and opportunity {}".format(_chronicle['event'].satellite, _chronicle['event'].opportunity))
        # if type(_chronicle['event'])==CommunicationEvent:
        #     print("Communication: station {} to sat {} during pass {}".format(_chronicle['event'].station, _chronicle['event'].satellite, _chronicle['event'].comm_pass))

In [111]:
# plot_history(world_with_brokers)
# # for i in range(10):
#     # plot_event(world_with_brokers.history[i], world_with_brokers)
